# DQI-Kit: Hands-on Exercises

### DQI-Kit Cheat Sheet
| You want to ... | Use |
| --- | --- |
| state a Max-XORSAT / Max-LINSAT instance directly | `MaxXorSat()`, `MaxLinSat(GF(p))`, `prob.new_var(name)`, `prob.add_constraint(x + y == 1)`, `prob.add_constraint(x != y)` |
| look at the instance | `prob.get_B()`, `prob.get_v()`, `prob.get_F()`, `prob.get_m()`, `prob.get_n()`, `prob.get_minimum_distance()` |
| describe a problem with integer/binary variables, (in)equalities, Boolean formulas and objectives | `MaxConstraintSat()`, `new_var(name, lo, hi)`, `new_binary_var(name)`, `add_constraint(2 * x + y <= 5, weight=w)`, `add_boolean_constraint(a \| b)`, `add_objective(expr, minimize=False)` |
| turn that into Max-LINSAT | `prob.to_max_linsat(max_degree=3, merge_strategy=MergeStrategy.DUPLICATES, var_range_constraint_factor=...)` |
| estimate what DQI achieves | `Dqi(linsat).estimate_solution_quality(l, details=True)` → `.value`, `.l`, `.method`, `.guaranteed` |
| compare with classical solvers | `BruteForceSolver`, `SimAnnealSolver`, `PrangeSolver` (each with `.get_solution_quality()`), `linsat.get_optimal_solution_quality()` (OR-Tools), `linsat.get_random_solution_value()` |

**Run the setup cell below first.** 

In [ ]:
import itertools
import warnings

import networkx as nx
from sage.all import GF, codes, vector
from sage.coding.decoder import DecodingError

from classical_solvers import BruteForceSolver, PrangeSolver, SimAnnealSolver
from decoders import SyndromeDecoder
from dqi import Dqi
from max_constraint_sat import MaxConstraintSat
from max_lin_sat import MaxLinSat, MaxXorSat, MergeStrategy, OptimalPolynomialIntersection, VertexColor

# DQI-Kit warns about unused variables, m <= n, ...; that is just noise for these exercises.
warnings.filterwarnings("ignore")

# Part I: Coding theory (Sage only)

## Exercise 1: Decoding the Hamming code

*Syndrome decoding by hand and in Sage.* 

1. Build the $[7, 4]$ Hamming code in Sage and look at $\mathbf{G}$, $\mathbf{H}$ and the minimum distance.
2. Flip one bit of a codeword and compute the syndrome $\mathbf{s} = \mathbf{H}\tilde{\mathbf{c}}$. Which column of $\mathbf{H}$ is it? Decode.
3. Flip two bits: what happens to the decoded word?

**Task 1.** Sage builds the Hamming code with $r$ parity checks from `codes.HammingCode(GF(2), r)`.

In [ ]:
C = codes.HammingCode(GF(2), 3)  # r = 3 parity checks  ->  the [7, 4] Hamming code
G = ...  # generator matrix of C
H = ...  # parity-check matrix of C

print(C)
print("G =", G, sep="\n")
print("H =", H, sep="\n")
print("minimum distance d =", ...)

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
C = codes.HammingCode(GF(2), 3)  # r = 3 parity checks  ->  the [7, 4] Hamming code
G = C.generator_matrix()
H = C.parity_check_matrix()

print(C)
print("G =", G, sep="\n")
print("H =", H, sep="\n")
print("minimum distance d =", C.minimum_distance())
```

</details>

**Task 2.** Compare the syndrome with the columns of $\mathbf{H}$: the position of the matching column is the error position. Sage's syndrome decoder does exactly this lookup.

In [ ]:
c = C.random_element()
e = vector(GF(2), [0, 0, 0, 0, 1, 0, 0])  # error pattern: flip bit 4 (positions count from 0)
received = c + e

s = ...  # the syndrome H * received
print("codeword  c  =", c)
print("received  c~ =", received)
print("syndrome  s  =", s, " = column(s)", [j for j in range(7) if H.column(j) == s], "of H")

decoded = ...  # C.decoder("Syndrome").decode_to_code(...)
print("decoded      =", decoded, "  correct:", decoded == c)

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
c = C.random_element()
e = vector(GF(2), [0, 0, 0, 0, 1, 0, 0])  # error pattern: flip bit 4 (positions count from 0)
received = c + e

s = H * received
print("codeword  c  =", c)
print("received  c~ =", received)
print("syndrome  s  =", s, " = column(s)", [j for j in range(7) if H.column(j) == s], "of H")

decoded = C.decoder("Syndrome").decode_to_code(received)
print("decoded      =", decoded, "  correct:", decoded == c)
```

</details>

**Task 3.** Now flip two bits. The decoder still returns *a* codeword. Which one, and how far is it from the original?

In [ ]:
e2 = vector(GF(2), [1, 0, 0, 0, 1, 0, 0])  # two flipped bits
received2 = c + e2

decoded2 = ...  # decode received2
print("decoded  =", decoded2, "  correct:", decoded2 == c)
print("distance between decoded word and original codeword:", (decoded2 - c).hamming_weight())

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
e2 = vector(GF(2), [1, 0, 0, 0, 1, 0, 0])  # two flipped bits
received2 = c + e2

decoded2 = C.decoder("Syndrome").decode_to_code(received2)
print("decoded  =", decoded2, "  correct:", decoded2 == c)
print("distance between decoded word and original codeword:", (decoded2 - c).hamming_weight())
```

</details>

**Lessons learned**
- $d = 3$ corrects $\lfloor (d-1)/2 \rfloor = 1$ errors.
- With two errors the received word is closer to a different codeword (at distance 3 from the original).

## Exercise 2: Decoding Reed-Solomon codes

*Interpolation is decoding: see how many errors a Reed-Solomon code tolerates.*

1. Encode the polynomial $2 + 5x + x^2$ with a Reed-Solomon code over $\mathbb{F}_7$ ($k = 3$, $n = 6$).
2. Add one, two, three errors and decode: when does Berlekamp-Welch fail?
3. Lower $k$ to 2: how many errors can be corrected now?

**Task 1.** A Reed-Solomon codeword is the list of values $(f(1), \dots, f(6))$ of a polynomial $f$ of degree $< k$. The message is the coefficient vector of $f$.

In [ ]:
F = GF(7)
points = [F(i) for i in range(1, 7)]              # evaluation points 1, ..., 6  ->  n = 6
C = codes.GeneralizedReedSolomonCode(points, 3)   # k = 3: polynomials of degree < 3

message = vector(F, [2, 5, 1])  # coefficients of f(x) = 2 + 5x + x^2
c = ...  # encode the message: c = (f(1), ..., f(6))

print(C)
print("codeword c =", c)
print("minimum distance d = n - k + 1 =", C.minimum_distance())

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
F = GF(7)
points = [F(i) for i in range(1, 7)]              # evaluation points 1, ..., 6  ->  n = 6
C = codes.GeneralizedReedSolomonCode(points, 3)   # k = 3: polynomials of degree < 3

message = vector(F, [2, 5, 1])  # coefficients of f(x) = 2 + 5x + x^2
c = C.encode(message)

print(C)
print("codeword c =", c)
print("minimum distance d = n - k + 1 =", C.minimum_distance())
```

</details>

**Task 2.** The helper corrupts the first `n_errors` positions of a codeword. Complete the decoding call (`BerlekampWelch` decoder) and see for which number of errors the original codeword comes back.

In [ ]:
def decode_with_errors(C, c, n_errors):
    """Corrupt n_errors positions of c and decode; True if c is recovered."""
    received = vector(C.base_field(), list(c))  # a copy of c
    for i in range(n_errors):
        received[i] += 3
    try:
        decoded = ...  # decode `received` with C.decoder("BerlekampWelch")
        return decoded == c
    except (DecodingError, ValueError):
        return False  # the decoder gave up


for n_errors in (1, 2, 3):
    print(f"{n_errors} error(s): decoded correctly = {decode_with_errors(C, c, n_errors)}")
print("unique decoding radius (d - 1) // 2 =", (C.minimum_distance() - 1) // 2)

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def decode_with_errors(C, c, n_errors):
    """Corrupt n_errors positions of c and decode; True if c is recovered."""
    received = vector(C.base_field(), list(c))  # a copy of c
    for i in range(n_errors):
        received[i] += 3
    try:
        decoded = C.decoder("BerlekampWelch").decode_to_code(received)
        return decoded == c
    except (DecodingError, ValueError):
        return False  # the decoder gave up


for n_errors in (1, 2, 3):
    print(f"{n_errors} error(s): decoded correctly = {decode_with_errors(C, c, n_errors)}")
print("unique decoding radius (d - 1) // 2 =", (C.minimum_distance() - 1) // 2)
```

</details>

**Task 3.** Fewer coefficients means more redundancy. Build the code with $k = 2$ and repeat the experiment.

In [ ]:
C2 = codes.GeneralizedReedSolomonCode(points, ...)  # same points, k = 2
c2 = C2.encode(vector(F, [2, 5]))                    # f(x) = 2 + 5x

for n_errors in (1, 2, 3):
    print(f"k = 2, {n_errors} error(s): decoded correctly = {decode_with_errors(C2, c2, n_errors)}")
print("d =", C2.minimum_distance(), "  radius =", (C2.minimum_distance() - 1) // 2)

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
C2 = codes.GeneralizedReedSolomonCode(points, 2)  # same points, k = 2
c2 = C2.encode(vector(F, [2, 5]))                  # f(x) = 2 + 5x

for n_errors in (1, 2, 3):
    print(f"k = 2, {n_errors} error(s): decoded correctly = {decode_with_errors(C2, c2, n_errors)}")
print("d =", C2.minimum_distance(), "  radius =", (C2.minimum_distance() - 1) // 2)
```

</details>

**Lessons learned:**
- For Reed-Solomon codes, $d = n - k + 1$,
- Berlekamp-Welch therefore corrects exactly $\lfloor (n-k)/2 \rfloor$ errors: one for $k = 3$, two for $k = 2$.

# Part II: The DQI algorithm

## Exercise 3: Max-Cut with DQI

*From a graph to a Max-XORSAT instance and through DQI.* 

1. Encode the graph below as Max-XORSAT: one constraint $x_u + x_v = 1$ per edge. Look at $\mathbf{B}$ and $\mathbf{v}$.
2. Run DQI with $\ell = 1$ and compare with the maximum cut (brute force) and with random guessing.
3. Remove the two edges that close triangles, $(2, 3)$ and $(4, 5)$: how do $\Delta_{\min}$ and the DQI estimate change?

This is the six-vertex graph from the slides; exercise 4 uses it again.

In [ ]:
EDGES = [(1, 2), (1, 3), (2, 3), (2, 4), (3, 4), (3, 5), (4, 5), (4, 6), (5, 6)]
POS = {1: (0, 0.5), 2: (1, 1), 3: (1, 0), 4: (2, 1), 5: (2, 0), 6: (3, 0.5)}

G = nx.Graph(EDGES)
nx.draw(G, POS, with_labels=True, node_color="lightsteelblue", node_size=700, font_weight="bold")

**Task 1.** A vertex $u$ is on side $x_u \in \{0, 1\}$ of the cut; an edge is cut iff its endpoints are on different sides. Add that constraint for every edge.

In [ ]:
def max_cut(G):
    prob = MaxXorSat()
    x = ... # one F_2 variable per vertex: prob.new_var([var. name])
    for u, v in G.edges:
        ...  # edge (u, v) is cut  <=>  x_u + x_v = 1
    return prob


prob = max_cut(G)
print(prob)
print("B =", prob.get_B(), sep="\n")
print("v =", prob.get_v())

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def max_cut(G):
    prob = MaxXorSat()
    x = {u: prob.new_var(f"x_{u}") for u in sorted(G)}  # one F_2 variable per vertex
    for u, v in G.edges:
        prob.add_constraint(x[u] + x[v] == 1)  # edge (u, v) is cut  <=>  x_u + x_v = 1
    return prob


prob = max_cut(G)
print(prob)
print("B =", prob.get_B(), sep="\n")
print("v =", prob.get_v())
```

</details>

**Task 2.** Complete the report: the DQI estimate for $\ell = 1$ (ask for `details=True` to also see which estimator was used) and the optimum by brute force. `report` is reused in exercise 4.

In [ ]:
def report(prob):
    m = prob.get_m()
    est = ...  # Dqi(prob).estimate_solution_quality(...) with l = 1 and details=True
    opt = ...  # BruteForceSolver(prob).get_solution_quality()
    print(f"m = {m} constraints, d_min = {prob.get_minimum_distance()}")
    print(f"  DQI with l = 1 ({est.method}): {est.value:.3f} constraints satisfied in expectation")
    print(f"  random guessing: {prob.get_random_solution_value():.3f}     optimum: {opt}")


report(prob)

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def report(prob):
    m = prob.get_m()
    est = Dqi(prob).estimate_solution_quality(1, details=True)
    opt = BruteForceSolver(prob).get_solution_quality()
    print(f"m = {m} constraints, d_min = {prob.get_minimum_distance()}")
    print(f"  DQI with l = 1 ({est.method}): {est.value:.3f} constraints satisfied in expectation")
    print(f"  random guessing: {prob.get_random_solution_value():.3f}     optimum: {opt}")


report(prob)
```

</details>

**Task 3.** Remove the edges $(2, 3)$ and $(4, 5)$. The graph becomes bipartite, so every edge can be cut.

In [ ]:
G2 = G.copy()
G2.remove_edges_from(...)  # the two triangle-closing edges
report(max_cut(G2))

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
G2 = G.copy()
G2.remove_edges_from([(2, 3), (4, 5)])  # the two triangle-closing edges
report(max_cut(G2))
```

</details>

**Lessons Learned**
- Triangles are linear dependencies among three rows of $\mathbf{B}$, so $\Delta_{\min} = 3$.
- Without them $\Delta_{\min}$ rises to 4, the $\ell = 1$ estimate becomes the guaranteed closed form, and DQI's fraction of cut edges improves.
- Random guessing cuts half the edges.

## Exercise 4: 3-COLOR with DQI

*Same graph, three colours: from Max-XORSAT to Max-LINSAT with set constraints.*

1. Switch your Max-Cut code to $\mathbb{F}_3$ and $\neq$ constraints. Look at the sets $F_i$.
2. Run DQI: how many edges are properly coloured in expectation? Compare with the optimum.
3. Add edges until the graph is no longer 3-colourable: what does DQI return now?

**Task 1.** Colours are elements of $\mathbb{F}_3$. An edge is properly coloured iff $x_u \neq x_v$, i.e. $x_u - x_v \in \{1, 2\}$: a Max-LINSAT constraint with a *set* on the right-hand side.

In [ ]:
def three_color(G):
    prob = MaxLinSat(GF(3))  # one colour in {0, 1, 2} per vertex
    x = {u: prob.new_var(f"x_{u}") for u in sorted(G)}
    for u, v in G.edges:
        ...  # edge (u, v) is properly coloured  <=>  x_u != x_v
    return prob


prob = three_color(G)
print(prob)
print("B =", prob.get_B(), sep="\n")
print("F =", prob.get_F())

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def three_color(G):
    prob = MaxLinSat(GF(3))  # one colour in {0, 1, 2} per vertex
    x = {u: prob.new_var(f"x_{u}") for u in sorted(G)}
    for u, v in G.edges:
        prob.add_constraint(x[u] != x[v])  # edge (u, v) is properly coloured  <=>  x_u != x_v
    return prob


prob = three_color(G)
print(prob)
print("B =", prob.get_B(), sep="\n")
print("F =", prob.get_F())
```

</details>

**Task 2.** `report` from exercise 3 works unchanged. Note how random guessing moved from $m/2$ to $2m/3$.

In [ ]:
report(prob)

**Task 3.** Add edges so that vertices $1$ to $5$ contain a $K_4$; then the graph is not 3-colourable any more. `VertexColor(3, G)` is DQI-Kit's built-in version of `three_color`.

In [ ]:
G4 = G.copy()
G4.add_edges_from(...)  # e.g. (1, 4), (1, 5), (2, 5)
report(three_color(G4))

report(VertexColor(3, G4))  # same instance, built in

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
G4 = G.copy()
G4.add_edges_from([(1, 4), (1, 5), (2, 5)])  # vertices 1, 2, 3, 4, 5 now contain a K4
report(three_color(G4))

report(VertexColor(3, G4))  # same instance, built in
```

</details>

**Lessons learned:**
- Nothing in DQI cares whether the instance is satisfiable.
- It returns the expected number of satisfied constraints, which now falls short of $m$ just as the optimum does.
- It still beats random guessing.

# Part III: DQI on structured problems

## Exercise 5: Optimal Polynomial Intersection

*DQI with a Reed-Solomon decoder: a closed-form estimate, no decoding needed.*

1. Generate a random OPI instance over $\mathbb{F}_{31}$ with $|F_i| = p/2$.
2. Run DQI and compare with simulated annealing and Prange's algorithm.
3. Vary the degree: how does the DQI ratio change with $n/p$?

**Task 1.** OPI asks for a polynomial of degree $<n$ over $\mathbb{F}_p$ whose value at each $x \in \mathbb{F}_p$ lies in a random set $F_x$ of size $r$. `OptimalPolynomialIntersection.random(F, degree, r)` draws such an instance.

In [ ]:
p = 31
F = GF(p)
opi = OptimalPolynomialIntersection.random(F, degree=9, r=...)  # |F_i| = r = p // 2

m, n = opi.get_m(), opi.get_n()
print(f"p = {p}, n = {n} coefficients, m = {m} constraints, |F_i| = {len(opi.get_F()[0])}")
print("minimum distance d =", opi.get_minimum_distance())

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
p = 31
F = GF(p)
opi = OptimalPolynomialIntersection.random(F, degree=9, r=p // 2)  # |F_i| = r = p // 2

m, n = opi.get_m(), opi.get_n()
print(f"p = {p}, n = {n} coefficients, m = {m} constraints, |F_i| = {len(opi.get_F()[0])}")
print("minimum distance d =", opi.get_minimum_distance())
```

</details>

**Task 2.** Without an explicit $\ell$, DQI-Kit picks the best degree the Reed-Solomon decoder can still handle. Compare with `SimAnnealSolver(opi, steps=2000)` and with Prange's expected value.

In [ ]:
est = Dqi(opi).estimate_solution_quality(details=True)
print(f"DQI (l = {est.l}, {est.method}): {est.value:.2f} / {m}")

sim_anneal = ...  # SimAnnealSolver(opi, steps=2000).get_solution_quality()
prange = ...  # PrangeSolver(opi).get_expected_solution_quality()
print(f"simulated annealing: {sim_anneal} / {m}")
print(f"Prange (expected):   {prange:.2f} / {m}")

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
est = Dqi(opi).estimate_solution_quality(details=True)
print(f"DQI (l = {est.l}, {est.method}): {est.value:.2f} / {m}")

sim_anneal = SimAnnealSolver(opi, steps=2000).get_solution_quality()
prange = PrangeSolver(opi).get_expected_solution_quality()
print(f"simulated annealing: {sim_anneal} / {m}")
print(f"Prange (expected):   {prange:.2f} / {m}")
```

</details>

**Task 3.** More coefficients ($n$ closer to $p$) mean a smaller code and a stronger decoder. Fill in the DQI estimate.

In [ ]:
for degree in (4, 9, 14, 19):
    inst = OptimalPolynomialIntersection.random(F, degree=degree, r=p // 2)
    dqi = ...  # DQI estimate for inst (plain value is enough)
    prange = PrangeSolver(inst).get_expected_solution_quality()
    print(f"n/p = {inst.get_n() / p:.2f}:   DQI {dqi / m:.3f}   Prange {prange / m:.3f}")

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
for degree in (4, 9, 14, 19):
    inst = OptimalPolynomialIntersection.random(F, degree=degree, r=p // 2)
    dqi = Dqi(inst).estimate_solution_quality()
    prange = PrangeSolver(inst).get_expected_solution_quality()
    print(f"n/p = {inst.get_n() / p:.2f}:   DQI {dqi / m:.3f}   Prange {prange / m:.3f}")
```

</details>

**Lessons learned:**
- For OPI, the DQI code is a Reed-Solomon code, so the decoder is Berlekamp-Welch.
- The ratio follows the semicircle law from the slides and grows with $n/p$.

# Part IV: Encoding problems for DQI

## Exercise 6: Production planning

*A real use case: linear capacity constraints and a linear profit objective.*

> **Use case.** A workshop builds **chairs** (2 h of work, 1 unit of wood, profit 2) and **tables** (3 h, 2 units of wood, profit 3). 8 h and 5 units of wood are available; at most 3 chairs and 2 tables fit into the shop. Maximise the profit.

1. Encode: two integer variables, two $\le$ constraints (weight $>$ maximum profit), one objective.
2. Convert: which $p$, $n$, $m$? Which rows encode the objective, and why so many duplicates?
3. Compare the Max-LINSAT optimum with the best production plan and with the DQI estimate.

**Task 1.** `new_var(name, lo, hi)` creates an integer variable with a range. Breaking a capacity has to cost more than any profit could gain, so give the $\le$ constraints a weight above the maximum profit.

In [ ]:
MAX_PROFIT = 2 * 3 + 3 * 2  # everything that fits into the shop is sold

prob = MaxConstraintSat()
chairs = prob.new_var("chairs", 0, 3)  # integer variable with range [0, 3]
tables = ...  # integer variable with range [0, 2]
prob.add_constraint(2 * chairs + 3 * tables <= 8, weight=MAX_PROFIT + 1)  # labour
...  # wood: 1 unit per chair, 2 per table, 5 available; same weight
...  # objective: the profit

for c in prob.constraints:
    print("constraint:", c)
for o in prob.objectives:
    print("objective: ", o)

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
MAX_PROFIT = 2 * 3 + 3 * 2  # everything that fits into the shop is sold

prob = MaxConstraintSat()
chairs = prob.new_var("chairs", 0, 3)  # integer variable with range [0, 3]
tables = prob.new_var("tables", 0, 2)  # integer variable with range [0, 2]
prob.add_constraint(2 * chairs + 3 * tables <= 8, weight=MAX_PROFIT + 1)  # labour
prob.add_constraint(chairs + 2 * tables <= 5, weight=MAX_PROFIT + 1)  # wood
prob.add_objective(2 * chairs + 3 * tables)  # profit

for c in prob.constraints:
    print("constraint:", c)
for o in prob.objectives:
    print("objective: ", o)
```

</details>

**Task 2.** Convert and read the result: What is $p$, and why? Which rows come from the objective? Where do the multiplicities (`(x13)`, `(x54)`, ...) come from?

In [ ]:
linsat = prob.to_max_linsat()
p, n, m = linsat.field.order(), linsat.get_n(), linsat.get_m()
print(f"field F_{p},  n = {n} variables,  m = {m} rows for {len(linsat.constraints)} distinct constraints:")
for c in linsat.constraints.values():
    print("  ", str(c).replace("\n", " | "))

<details>
<summary><b>Answer</b> (click to expand)</summary>

* $p = 13$ is the smallest prime above the largest value any expression can take ($2 \cdot 3 + 3 \cdot 2 = 12$), so that arithmetic never wraps around.
* The rows `chairs = 0 | 1 | 2 | 3` and `tables = 0 | 1 | 2` encode the ranges of the variables **and** the objective: a linear objective is a set of weighted equality constraints, one per value, weighted by how much profit that value earns (plus a large base weight that keeps the variable in range).
* Weights become duplicate rows (`MergeStrategy.DUPLICATES`), so $m \approx 600$ although there are only 5 distinct constraints.

</details>

**Task 3.** `get_optimal_solution()` solves the Max-LINSAT instance with OR-Tools. With this many rows the default decoder is slow, so pass `SyndromeDecoder.constructor()` to `Dqi`.

In [ ]:
x = linsat.get_optimal_solution()
n_chairs, n_tables = (int(v) for v in x)
print(f"Max-LINSAT optimum: {linsat.get_optimal_solution_quality()} / {m} rows satisfied")
print(
    f"  -> {n_chairs} chairs, {n_tables} tables: profit {2 * n_chairs + 3 * n_tables},",
    f"labour {2 * n_chairs + 3 * n_tables} / 8 h, wood {n_chairs + 2 * n_tables} / 5",
)

est = Dqi(linsat, ...).estimate_solution_quality(details=True)  # use the SyndromeDecoder
print(f"DQI (l = {est.l}, {est.method}): {est.value:.2f} / {m}")
print(f"random guessing: {linsat.get_random_solution_value():.2f} / {m}")

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
x = linsat.get_optimal_solution()
n_chairs, n_tables = (int(v) for v in x)
print(f"Max-LINSAT optimum: {linsat.get_optimal_solution_quality()} / {m} rows satisfied")
print(
    f"  -> {n_chairs} chairs, {n_tables} tables: profit {2 * n_chairs + 3 * n_tables},",
    f"labour {2 * n_chairs + 3 * n_tables} / 8 h, wood {n_chairs + 2 * n_tables} / 5",
)

est = Dqi(linsat, SyndromeDecoder.constructor()).estimate_solution_quality(details=True)
print(f"DQI (l = {est.l}, {est.method}): {est.value:.2f} / {m}")
print(f"random guessing: {linsat.get_random_solution_value():.2f} / {m}")
```

</details>

**Lessons learned:**
- The Max-LINSAT optimum is the right production plan (1 chair, 2 tables, profit 8, both capacities exhausted).
- DQI, however, sits at the level of random guessing: with $\ell = 1$ out of $m \approx 600$ duplicated rows, a single correctable error is worth nothing.
- Weights are expensive for (unmodified) DQI.

## Exercise 7: Boolean constraints

*See how Boolean constraints turn into Max-XORSAT and why long clauses are expensive.*

1. Encode a clause $x_1 \lor x_2 \lor \dots \lor x_k$ for $k = 2, 3, 4, 5$ without a degree cap (`max_degree=k`).
2. Count the distinct Max-XORSAT constraints: how do they grow with $k$?
3. Cap `max_degree`: what happens to the number of variables and constraints?

**Task 1.** `new_binary_var` creates a Boolean variable, `|` builds the disjunction, `add_boolean_constraint` adds it. For $k = 2$: $x_0 \lor x_1 \mapsto x_0 + x_1 - x_0 x_1$, one Max-XORSAT constraint per monomial.

In [ ]:
def clause(k):
    """The clause x_0 | x_1 | ... | x_{k-1} as a MaxConstraintSat problem."""
    prob = MaxConstraintSat()
    x = [prob.new_binary_var(f"x_{i}") for i in range(k)]
    disjunction = x[0]
    for x_i in x[1:]:
        disjunction = disjunction | x_i
    ...  # add the disjunction as a Boolean constraint
    return prob


xorsat = clause(2).to_max_linsat()
print("x_0 | x_1  ->  x_0 + x_1 - x_0 x_1  ->")
for c in xorsat.constraints.values():
    print("  ", str(c).replace("\n", " | "))

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def clause(k):
    """The clause x_0 | x_1 | ... | x_{k-1} as a MaxConstraintSat problem."""
    prob = MaxConstraintSat()
    x = [prob.new_binary_var(f"x_{i}") for i in range(k)]
    disjunction = x[0]
    for x_i in x[1:]:
        disjunction = disjunction | x_i
    prob.add_boolean_constraint(disjunction)
    return prob


xorsat = clause(2).to_max_linsat()
print("x_0 | x_1  ->  x_0 + x_1 - x_0 x_1  ->")
for c in xorsat.constraints.values():
    print("  ", str(c).replace("\n", " | "))
```

</details>

**Task 2.** Convert without a degree cap, i.e. `max_degree=k`, and count the distinct constraints (the range constraints $x_i \in \{0, 1\}$ are trivially true over $\mathbb{F}_2$ and are dropped).

In [ ]:
for k in (2, 3, 4, 5):
    xorsat = clause(k).to_max_linsat(max_degree=...)  # no degree cap
    print(f"k = {k}: {len(xorsat.constraints)} distinct constraints,  n = {xorsat.get_n()} variables")

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
for k in (2, 3, 4, 5):
    xorsat = clause(k).to_max_linsat(max_degree=k)  # no degree cap
    print(f"k = {k}: {len(xorsat.constraints)} distinct constraints,  n = {xorsat.get_n()} variables")
```

</details>

**Task 3.** Keep $k = 5$ and lower `max_degree` step by step. Watch both $n$ and the number of constraints.

In [ ]:
for max_degree in (5, 4, 3, 2):
    xorsat = clause(5).to_max_linsat(max_degree=max_degree)
    print(f"k = 5, max_degree = {max_degree}: {len(xorsat.constraints)} distinct constraints,  n = {xorsat.get_n()} variables")

**Lessons learned:**
- A $k$-clause has $2^k - 1$ monomials and therefore $2^k - 1$ Max-XORSAT constraints.
- Capping the degree trades constraints for auxiliary variables (and equality constraints that tie them to the products they stand for); it does not make long clauses cheap.

## Exercise 8: Shift planning

*A real use case whose natural encoding has short linear dependencies.*

> **Use case.** Four workers each take the **morning** (0) or the **evening** (1) shift. Alice/Bob, Bob/Carol and Alice/Carol must not share a shift, Dave works mornings, and Carol or Dave must cover the evening.

1. Encode with binary variables, convert to Max-XORSAT.
2. Compute $\Delta_{\min}$ and find the shortest set of dependent rows of $\mathbf{B}$: which pattern is it, where does it come from?
3. Convert with `USE_STRICTEST`, then drop the Alice/Carol conflict: which pattern shows up next?

**Task 1.** Conflicts are `!=` constraints, Dave's shift is an `==` constraint, and the coverage rule is a Boolean `|`.

In [ ]:
def shift_plan(alice_carol_conflict=True):
    prob = MaxConstraintSat()
    alice, bob, carol, dave = (prob.new_binary_var(name) for name in ["alice", "bob", "carol", "dave"])
    prob.add_constraint(alice != bob)
    ...  # Bob and Carol must not share a shift
    if alice_carol_conflict:
        ...  # Alice and Carol must not share a shift
    ...  # Dave works mornings
    ...  # Carol or Dave covers the evening (Boolean constraint)
    return prob


linsat = shift_plan().to_max_linsat()
print(f"Max-XORSAT with n = {linsat.get_n()} variables and m = {linsat.get_m()} rows; distinct constraints:")
for c in linsat.constraints.values():
    print("  ", str(c).replace("\n", " | "))

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def shift_plan(alice_carol_conflict=True):
    prob = MaxConstraintSat()
    alice, bob, carol, dave = (prob.new_binary_var(name) for name in ["alice", "bob", "carol", "dave"])
    prob.add_constraint(alice != bob)
    prob.add_constraint(bob != carol)
    if alice_carol_conflict:
        prob.add_constraint(alice != carol)
    prob.add_constraint(dave == 0)
    prob.add_boolean_constraint(carol | dave)
    return prob


linsat = shift_plan().to_max_linsat()
print(f"Max-XORSAT with n = {linsat.get_n()} variables and m = {linsat.get_m()} rows; distinct constraints:")
for c in linsat.constraints.values():
    print("  ", str(c).replace("\n", " | "))
```

</details>

**Task 2.** The helper searches for the fewest rows of $\mathbf{B}$ that are linearly dependent (their number is $\Delta_{\min}$) and prints them as constraints. Fill in the minimum distance and explain the pattern you see.

In [ ]:
def shortest_dependency(linsat, max_len=4):
    """Indices of the fewest linearly dependent rows of B."""
    B = linsat.get_B()
    for size in range(1, max_len + 1):
        for rows in itertools.combinations(range(B.nrows()), size):
            if B.matrix_from_rows(rows).rank() < size:
                return list(rows)
    return None


def show(name, linsat):
    B, F = linsat.get_B(), linsat.get_F()
    names = [v.name for v in linsat.variables]
    d_min = ...  # the minimum distance of linsat
    print(f"{name}: n = {linsat.get_n()}, m = {linsat.get_m()}, d_min = {d_min}")
    rows = shortest_dependency(linsat)
    print("  shortest dependency: rows", rows)
    for i in rows:
        terms = " + ".join(f"{c}*{v}" if c != 1 else v for c, v in zip(B.row(i), names) if c != 0)
        print(f"    {terms} in {F[i]}")


show("default conversion", linsat)

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def shortest_dependency(linsat, max_len=4):
    """Indices of the fewest linearly dependent rows of B."""
    B = linsat.get_B()
    for size in range(1, max_len + 1):
        for rows in itertools.combinations(range(B.nrows()), size):
            if B.matrix_from_rows(rows).rank() < size:
                return list(rows)
    return None


def show(name, linsat):
    B, F = linsat.get_B(), linsat.get_F()
    names = [v.name for v in linsat.variables]
    d_min = linsat.get_minimum_distance()
    print(f"{name}: n = {linsat.get_n()}, m = {linsat.get_m()}, d_min = {d_min}")
    rows = shortest_dependency(linsat)
    print("  shortest dependency: rows", rows)
    for i in rows:
        terms = " + ".join(f"{c}*{v}" if c != 1 else v for c, v in zip(B.row(i), names) if c != 0)
        print(f"    {terms} in {F[i]}")


show("default conversion", linsat)
```

</details>

<details>
<summary><b>Answer</b> (click to expand)</summary>

$\Delta_\min = 2$ because of **duplicate rows**. The Boolean OR is turned into Ising terms with weight $1/2$; to keep integer weights every other constraint is doubled, so each `!=` conflict appears twice.

</details>

**Task 3.** `MergeStrategy.USE_STRICTEST` drops the weights and keeps one row per constraint. Convert with it, then build the plan without the Alice/Carol conflict and look again.

In [ ]:
strict = shift_plan().to_max_linsat(merge_strategy=...)  # USE_STRICTEST
show("USE_STRICTEST", strict)

no_triangle = ...  # shift_plan(alice_carol_conflict=False) converted with USE_STRICTEST
show("USE_STRICTEST, no Alice/Carol conflict", no_triangle)

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
strict = shift_plan().to_max_linsat(merge_strategy=MergeStrategy.USE_STRICTEST)
show("USE_STRICTEST", strict)

no_triangle = shift_plan(alice_carol_conflict=False).to_max_linsat(merge_strategy=MergeStrategy.USE_STRICTEST)
show("USE_STRICTEST, no Alice/Carol conflict", no_triangle)
```

</details>

**Lessons learned:**
- Once the duplicates are gone, the Alice-Bob-Carol **triangle** of `!=` constraints is the shortest dependency ($\Delta_{\min} = 3$), exactly as in Max-Cut.
- Without the triangle the **AND/OR pattern** of `carol | dave` remains: the rows `carol`, `dave` and `carol + dave` are dependent, so $\Delta_{\min}$ stays at 3.
- Short dependencies are inherent to these encodings.

## Exercise 9: Dependency reduction

*Naïve vs. scalar vs. Vandermonde anchors on the three bad patterns.*

1. Build the three bad patterns over $\mathbb{F}_{23}$ (two duplicated rows, an AND/OR triple, a cycle): $\Delta_{\min}$?
2. Add $t = 3$ pins $a_k = 0$. Naïve (one row per pattern) and scalar anchors $\mu_r (a_1 + a_2 + a_3)$: how far does $\Delta_{\min}$ get?
3. Vandermonde anchors with fresh $\gamma_r$: $\Delta_{\min}$ for $t = 3, 5$? Compare the DQI estimates of all variants: why does $\Delta_{\min}$ barely matter here?

A *pinned* variable $a_k$ is $0$ at every optimum, so adding any multiple of it to the left-hand side of a row changes neither the optima nor the right-hand side, only $\mathbf{B}$. `build` assembles the instance from the patterns of task 1 plus `pins` pinned variables. An *anchor* is a function `anchor(r, i, a)` returning the term that is added to row $r$ (the $i$-th row of its pattern), given the list `a` of pinned variables. `GAMMAS` is the half-system $\{2, \dots, (p-1)/2\}$ of $\mathbb{F}_{23}$, one fresh $\gamma_r$ per row. Run this cell as it is.

In [ ]:
P = 23  # a safe prime: (P - 1) / 2 = 11 is prime, so the half-system has enough gammas and no gamma_r = -gamma_s
F = GF(P)
GAMMAS = [F(g) for g in range(2, (P - 1) // 2 + 1)]  # {2, ..., 11}: one fresh gamma_r for each of the 10 rows


def build(pins=0, anchor=None):
    """The three bad patterns over F_P plus `pins` pinned variables a_k = 0.

    anchor(r, i, a) returns the term added to the left-hand side of row r,
    the i-th row of its pattern; a is the list of pinned variables.
    """
    prob = MaxLinSat(F)
    x = [prob.new_var(f"x_{i}") for i in range(9)]
    a = [prob.new_var(f"a_{k}") for k in range(pins)]
    r = 0
    for group in bad_patterns(x):
        for i, (lhs, rhs) in enumerate(group):
            if anchor is not None:
                lhs = lhs + anchor(r, i, a)
            prob.add_constraint(lhs == rhs, disable_warnings=True)  # a_k = 0 at every optimum: rhs stays
            r += 1
    for a_k in a:
        prob.add_constraint(a_k == 0)  # the pins
    return prob


def evaluate(name, prob):
    d_min = prob.get_minimum_distance()  # takes a few seconds for the anchored variants
    est = Dqi(prob).estimate_solution_quality(details=True)
    print(
        f"{name:22s} n = {prob.get_n():2d}  m = {prob.get_m():2d}  d_min = {d_min}  "
        f"DQI (l = {est.l}) = {est.value:.2f}  random = {prob.get_random_solution_value():.2f}  "
        f"optimum = {prob.get_optimal_solution_quality()}"
    )

**Task 1.** Each pattern is a list of rows `(lhs, rhs)` standing for `lhs == rhs`. The two duplicated pairs are given. Add the AND/OR triple from exercise 8 ($x_4 = 1$, $x_5 = 1$, $x_4 + x_5 = 2$) and a cycle ($x_6 - x_7 = 1$, $x_7 - x_8 = 1$, $x_6 - x_8 = 2$), then evaluate the instance without pins.

In [ ]:
def bad_patterns(x):
    """The rows of the instance, grouped by pattern; each row is (lhs, rhs) for lhs == rhs."""
    return [
        [(x[0] + x[1], 0), (x[0] + x[1], 0)],  # a constraint with weight 2 -> two identical rows
        [(x[2] + x[3], 0), (x[2] + x[3], 0)],
        [...],  # AND/OR triple: x_4 = 1, x_5 = 1, x_4 + x_5 = 2
        [...],  # cycle: x_6 - x_7 = 1, x_7 - x_8 = 1, x_6 - x_8 = 2
    ]


evaluate("no pins", build())

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def bad_patterns(x):
    """The rows of the instance, grouped by pattern; each row is (lhs, rhs) for lhs == rhs."""
    return [
        [(x[0] + x[1], 0), (x[0] + x[1], 0)],  # a constraint with weight 2 -> two identical rows
        [(x[2] + x[3], 0), (x[2] + x[3], 0)],
        [(x[4], 1), (x[5], 1), (x[4] + x[5], 2)],  # AND/OR triple
        [(x[6] - x[7], 1), (x[7] - x[8], 1), (x[6] - x[8], 2)],  # cycle
    ]


evaluate("no pins", build())
```

</details>

**Task 2.** Both variants use $t = 3$ pins and the sum $T = a_1 + a_2 + a_3$. *Naïve* (slide "Improve linear dependencies") adds $T$ to the first row of every pattern and nothing to the others. *Scalar anchors* add a different multiple $\mu_r T$ to every row; take $\mu_r$ = `GAMMAS[r]`. Write both as anchor functions.

In [ ]:
def naive(r, i, a):
    ...  # T = a_1 + ... + a_t on the first row of each pattern (i == 0), 0 on the others


def scalar(r, i, a):
    ...  # mu_r * T with a different mu_r = GAMMAS[r] for every row


evaluate("pin dilution", build(pins=3, anchor=dilution))
evaluate("scalar anchors", build(pins=3, anchor=scalar))

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def naive(r, i, a):
    return sum(a) if i == 0 else 0


def scalar(r, i, a):
    return GAMMAS[r] * sum(a)


evaluate("pin dilution", build(pins=3, anchor=dilution))
evaluate("scalar anchors", build(pins=3, anchor=scalar))
```

</details>

**Task 3.** Vandermonde anchors give row $r$ the coefficients $(\gamma_r, \gamma_r^2, \dots, \gamma_r^t)$ in front of $(a_1, \dots, a_t)$, i.e. they add $\sum_{k=1}^{t} \gamma_r^k a_k$. Run $t = 3$ and $t = 5$ and compare $\Delta_{\min}$, $n$, $m$ and the DQI estimates of all variants.

In [ ]:
def vandermonde(r, i, a):
    ...  # gamma_r * a_1 + gamma_r^2 * a_2 + ... + gamma_r^t * a_t with gamma_r = GAMMAS[r]


evaluate("Vandermonde, t = 3", build(pins=3, anchor=vandermonde))
evaluate("Vandermonde, t = 5", build(pins=5, anchor=vandermonde))

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def vandermonde(r, i, a):
    return sum(GAMMAS[r] ** (k + 1) * a_k for k, a_k in enumerate(a))


evaluate("Vandermonde, t = 3", build(pins=3, anchor=vandermonde))
evaluate("Vandermonde, t = 5", build(pins=5, anchor=vandermonde))
```

</details>

**Lessons learned:**
- Naïve and scalar anchors get stuck at $\Delta_{\min} = 4$ no matter how many pins are added: the difference of two rows of the same pattern is a multiple of $T$, and two such differences cancel each other.
- Vandermonde anchors with a fresh $\gamma_r$ per row reach $\Delta_{\min} = t + 2$ for the price of $t$ extra variables and $t$ extra rows. The costs are a dense $\mathbf{B}$ and a field large enough to supply one $\gamma_r$ per row.
- The DQI estimates barely move: with $r / p = 1 / 23$ every variant sits far left in the semicircle law, so the tiny sets $F_i$, not $\Delta_{\min}$, are the bottleneck here.